##### ARTI 560 - Computer Vision

## Visual Representations with DINOv2 - Exercise

### Exercise 1: Unsupervised Clustering

In this exercise, you will use the `KMeans` algorithm from sklearn to group 20 images from the Oxford Pet dataset into 2 clusters (Cats vs. Dogs) based purely on their CLS tokens.

Instructions:

1.  Extract the 384-dimensional [CLS] tokens from 20 images of the Oxford-IIIT Pet dataset. Ensure your selection includes a mix of both cats and dogs.

2. Apply K-Means Clustering ($n=2$) to group the vectors based on mathematical similarity rather than provided labels.

3. Compare the predicted clusters against ground-truth labels.

In [ ]:
import os
import numpy as np
from PIL import Image
from sklearn.cluster import KMeans
from sklearn.metrics import accuracy_score, confusion_matrix
import torch
from transformers import AutoImageProcessor, AutoModel

# ------------------------------
# ------------------------------
images_dir = r"C:\Users\A NWAR DAHAN\Downloads\lab10-visual-representations\images"
annotations_dir = r"C:\Users\A NWAR DAHAN\Downloads\lab10-visual-representations\annotations"

list_file = os.path.join(annotations_dir, "list")   # renamed annotations.txt
trainval_file = os.path.join(annotations_dir, "trainval")

# ------------------------------
# ------------------------------
model_id = "facebook/dinov2-small"
processor = AutoImageProcessor.from_pretrained(model_id)
model = AutoModel.from_pretrained(model_id)
model.eval()

# ------------------------------
# ------------------------------
species_dict = {}   # img_name (without .jpg) -> 'cat' or 'dog'
with open(list_file, 'r') as f:
    for line in f:
        parts = line.strip().split()
        if len(parts) < 2:
            continue
        img_id = parts[0]          # e.g., "Abyssinian_100"
        species = int(parts[1])    # 1 or 2
        label = "cat" if species == 1 else "dog"
        species_dict[img_id] = label

# ------------------------------
# ------------------------------
with open(trainval_file, 'r') as f:
    trainval_ids = [line.strip() for line in f if line.strip()]

cat_ids = [img_id for img_id in trainval_ids if species_dict.get(img_id) == "cat"]
dog_ids = [img_id for img_id in trainval_ids if species_dict.get(img_id) == "dog"]

selected_ids = cat_ids[:10] + dog_ids[:10]
print(f"Selected {len(selected_ids)} images: {len(cat_ids[:10])} cats, {len(dog_ids[:10])} dogs")

# ------------------------------
# ------------------------------
cls_tokens = []
true_labels = []

for img_id in selected_ids:
    img_path = os.path.join(images_dir, img_id + ".jpg")
    if not os.path.exists(img_path):
        print(f"Warning: {img_path} not found – skipping.")
        continue
    image = Image.open(img_path).convert("RGB")
    inputs = processor(images=image, return_tensors="pt")
    with torch.no_grad():
        outputs = model(**inputs)
    cls = outputs.last_hidden_state[0, 0, :].cpu().numpy()
    cls_tokens.append(cls)
    true_labels.append(species_dict[img_id])

cls_tokens = np.array(cls_tokens)
true_labels = np.array(true_labels)

# ------------------------------
# ------------------------------
kmeans = KMeans(n_clusters=2, random_state=42, n_init=10)
pred_clusters = kmeans.fit_predict(cls_tokens)

cluster_to_label = {}
for cluster in [0, 1]:
    idx = np.where(pred_clusters == cluster)[0]
    if len(idx) == 0:
        continue
    cat_count = np.sum(true_labels[idx] == "cat")
    dog_count = len(idx) - cat_count
    cluster_to_label[cluster] = "cat" if cat_count > dog_count else "dog"

pred_labels = np.array([cluster_to_label[c] for c in pred_clusters])

# ------------------------------
# ------------------------------
accuracy = accuracy_score(true_labels, pred_labels)
print(f"\nClustering accuracy: {accuracy:.2f}")
print("Confusion matrix (rows=true, cols=pred):")
print(confusion_matrix(true_labels, pred_labels, labels=["cat", "dog"]))

misclustered = np.where(true_labels != pred_labels)[0]
if len(misclustered) > 0:
    print("\nMisclustered images:")
    for idx in misclustered:
        print(f"  {selected_ids[idx]}.jpg -> true={true_labels[idx]}, pred={pred_labels[idx]}")
else:
    print("\nPerfect clustering! All images correctly grouped.")

# ------------------------------
# ------------------------------
print("\n" + "="*60)
print("Comparison of predicted clusters against ground-truth labels")
print("="*60)
print(f"""
The K-means clustering using DINOv2's CLS tokens achieved an accuracy of {accuracy:.2%}.
{'All 20 images were correctly separated into cats and dogs.' if accuracy == 1.0 else f'There were {len(misclustered)} misclustered images.'}

Why do misclassifications happen? DINOv2 is a self-supervised vision transformer trained to group visually similar patches, not to distinguish cats from dogs. If a dog image has unusual features (e.g., a small, pointy‑eared breed like a Chihuahua that looks somewhat cat‑like) or a cat image has an unusual pose/background, the feature vectors may become more similar to the opposite class. Additionally, clustering is unsupervised – it finds any two natural groupings in the data, which may not always align perfectly with our human categories. With only 10 examples per class and no fine‑tuning, a few ambiguous samples can easily be swapped.
""")

### Exercise 2: Image Classification with DINOv2

In this exercise you'll use a DINOv2 model with a pre-trained linear head to classify an image. You will observe how the model maps visual features to specific ImageNet-1k categories.

Instructions:
1. For this exercise, you must use the following Model ID. This specific checkpoint includes the necessary classification head trained on ImageNet-1k:

    Model ID: `facebook/dinov2-small-imagenet1k-1-layer`

2. Find an image online to make the inference. To ensure the model has a fair chance of success, the image should belong to one of the ImageNet-1k classes (e.g., a Golden Retriever, a grand piano, a school bus, or a coffee mug).

In [ ]:
# Provide your solution here